# 02. Feature Engineering
We will extract temporal features (day, month, weekend flags) and time-series features (lags, rolling averages) to help our models capture historical dependencies.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../Dataset/raw/water_consumption_forecasting.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(by=["region", "date"]).reset_index(drop=True)

## 1. Temporal Features
Extracting month, day, day of week, and weekend indicator.

In [3]:
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
display(df[["date", "day_of_week", "is_weekend"]].head())

,date,day_of_week,is_weekend
0,2023-01-01,6,1
1,2023-01-02,0,0
2,2023-01-03,1,0
3,2023-01-04,2,0
4,2023-01-05,3,0


## 2. Time-Series Features (Lags & Rolling)
Creating features based on past consumption.

In [4]:
dfs = []
for region, group in df.groupby("region"):
    group = group.copy()
    group["lag_1"] = group["consumption_liters"].shift(1)
    group["lag_7"] = group["consumption_liters"].shift(7)
    group["rolling_mean_7"] = group["consumption_liters"].shift(1).rolling(window=7).mean()
    group["rolling_std_7"] = group["consumption_liters"].shift(1).rolling(window=7).std()
    dfs.append(group)

processed_df = pd.concat(dfs).dropna().reset_index(drop=True)
display(processed_df.head())

,region,date,consumption_liters,day_of_week,is_weekend,month,day,lag_1,lag_7,rolling_mean_7,rolling_std_7
0,Central,2023-01-08,15229.46,6,1,1,8,13596.90,14490.80,10624.100000,4925.797910
1,Central,2023-01-09,15022.47,0,0,1,9,15229.46,5916.42,10729.622857,5029.263310
2,Central,2023-01-10,12656.97,1,0,1,10,15022.47,18360.53,12030.487143,4746.521819
3,Central,2023-01-11,14112.66,2,0,1,11,12656.97,9499.28,11215.692857,3891.285699
4,Central,2023-01-12,14876.42,3,0,1,12,14112.66,5872.65,11874.747143,3942.471814


## 3. Train, Validation, Test Split
We will split the data chronologically: 70% Train, 15% Validation, 15% Test.

In [5]:
unique_dates = processed_df["date"].sort_values().unique()
n_dates = len(unique_dates)

train_idx = int(n_dates * 0.7)
val_idx = int(n_dates * 0.85)

train_dates = unique_dates[:train_idx]
val_dates = unique_dates[train_idx:val_idx]
test_dates = unique_dates[val_idx:]

train_df = processed_df[processed_df["date"].isin(train_dates)].reset_index(drop=True)
val_df = processed_df[processed_df["date"].isin(val_dates)].reset_index(drop=True)
test_df = processed_df[processed_df["date"].isin(test_dates)].reset_index(drop=True)

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

os.makedirs("../Dataset/processed", exist_ok=True)
processed_df.to_csv("../Dataset/processed/water_consumption_processed.csv", index=False)
train_df.to_csv("../Dataset/processed/train.csv", index=False)
val_df.to_csv("../Dataset/processed/val.csv", index=False)
test_df.to_csv("../Dataset/processed/test.csv", index=False)

Train size: 605
Val size: 130
Test size: 130
